In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece
!pip install -q datasets evaluate rouge_score textstat sacrebleu

In [ ]:
import torch
import re
import pandas as pd
import textstat

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

import evaluate

In [ ]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)


model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)


model.eval()

print("Model loaded successfully")

In [ ]:
def zero_shot_prompt(text):
    return f"""
Simplify the following text into plain language.

Return only the simplified sentence.

Text:
{text}

Simplified:
"""


def few_shot_prompt(text):
    return f"""
Simplify text into plain language.

Example:

Text:
The physician prescribed medication to alleviate symptoms.

Simplified:
The doctor gave medicine to reduce symptoms.


Text:
The committee postponed the implementation because of logistical problems.

Simplified:
The committee delayed the plan because of problems.


Text:
{text}

Simplified:
"""


def cot_prompt(text):
    return f"""
Think about how to simplify the following sentence.

Do not provide explanation.
Only output the simplified sentence.

Text:
{text}

Simplified:
"""

In [ ]:
def generate(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)


    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )


    generated = outputs[0][inputs.input_ids.shape[1]:]


    text = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )


    for stop in ["Human:", "Assistant:", "User:"]:
        if stop in text:
            text = text.split(stop)[0]


    return text.strip()

In [ ]:
asset = load_dataset(
    "facebook/asset",
    "simplification"
)

NUM_SAMPLES = 100

test_data = asset["test"].select(range(NUM_SAMPLES))

print("Total Samples:", len(test_data))

print("\nFirst Sample:")
print(test_data[0]["original"])

In [ ]:
zero_predictions = []
few_predictions = []
cot_predictions = []

sources = []
references = []

results = []


for i, sample in enumerate(test_data):

    source = sample["original"]
    reference = sample["simplifications"]


    zero_output = generate(
        zero_shot_prompt(source)
    )

    few_output = generate(
        few_shot_prompt(source)
    )

    cot_output = generate(
        cot_prompt(source)
    )


    sources.append(source)
    references.append(reference)

    zero_predictions.append(zero_output)
    few_predictions.append(few_output)
    cot_predictions.append(cot_output)


    results.append({
        "Original": source,
        "Reference": reference[0],
        "Zero Shot": zero_output,
        "Few Shot": few_output,
        "Chain of Thought": cot_output
    })


    print(f"{i+1}/100 completed")


results_df = pd.DataFrame(results)

print("Generation completed!")

In [ ]:
results_df.head(10)

In [ ]:
!pip install sacremoses
sari = evaluate.load("sari")
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

In [ ]:
references_text = []

for sample in test_data:
    references_text.append(
        sample["simplifications"]
    )

sources_text = [
    sample["original"]
    for sample in test_data
]

In [ ]:
def evaluate_model(predictions):

    # SARI
    sari_score = sari.compute(
        sources=sources_text,
        predictions=predictions,
        references=references_text
    )


    # BLEU
    bleu_score = bleu.compute(
        predictions=predictions,
        references=[
            [ref[0]]
            for ref in references_text
        ]
    )


    # ROUGE-L
    rouge_score = rouge.compute(
        predictions=predictions,
        references=[
            ref[0]
            for ref in references_text
        ]
    )


    # FKGL
    fkgl_scores = []

    for text in predictions:
        fkgl_scores.append(
            textstat.flesch_kincaid_grade(text)
        )


    return {
        "SARI": sari_score["sari"],
        "BLEU": bleu_score["bleu"],
        "ROUGE-L": rouge_score["rougeL"],
        "FKGL": sum(fkgl_scores)/len(fkgl_scores)
    }

In [ ]:
mistral_zero = evaluate_model(
    zero_predictions
)

mistral_few = evaluate_model(
    few_predictions
)

mistral_cot = evaluate_model(
    cot_predictions
)

In [ ]:
mistral_results = pd.DataFrame([
    {
        "Model":"Mistral-7B-Instruct",
        "Prompt":"Zero Shot",
        **mistral_zero
    },
    {
        "Model":"Mistral-7B-Instruct",
        "Prompt":"Few Shot",
        **mistral_few
    },
    {
        "Model":"Mistral-7B-Instruct",
        "Prompt":"Chain of Thought",
        **mistral_cot
    }
])


mistral_results